# ML Model Enhancement Analysis

## 🎯 Purpose
Analyze opportunities to enhance your breakout prediction models with new features from expanded data sources.

## 📊 Current Ensemble Architecture

Your current system uses a **2-layer ensemble**:

### **Layer 1: Base Models**
1. **Unified Model** (AUC: ~0.785)
   - Trained on all positions together
   - Best overall performer
   - Uses position encoding (is_rb, is_wr, is_te)

2. **Position-Specific Models** (3 models)
   - RB Model - Optimized for rushing volume & snap share
   - WR Model - Optimized for targets & routes
   - TE Model - Optimized for blocking vs receiving role

### **Layer 2: Ensemble Strategy**
- **Best performer:** 95% Unified + 5% Position-Specific
- Tests multiple strategies: simple average, performance-weighted, confidence-weighted
- Final output: Ensemble prediction combining both model types

## 🚀 Enhancement Goals
1. **Improve base model features** - Add contextual & situational features
2. **Enhance position-specific models** - Add position-relevant features (RZ for RBs, routes for WRs)
3. **Boost ensemble performance** - Better model diversity through richer features
4. **Increase recall** - Catch more breakouts 1-2 weeks early (target: 80%+)

## 📋 Current Features (13 total)

### **Usage Metrics** (6 features)
* `snap_share` - % of offensive snaps played
* `snap_share_delta` - Week-over-week snap % change **⭐ Top predictor (27% importance)**
* `touches` - Rushing attempts + receptions
* `touches_delta` - Week-over-week touch change
* `targets` - Pass targets (recently added)
* `targets_delta` - Week-over-week target change

### **Historical Context** (3 features)
* `avg_snap_share_prev_2wk` - Rolling 2-week snap average
* `avg_fantasy_points_prev_2wk` - Rolling 2-week fantasy average
* `avg_targets_prev_2wk` - Rolling 2-week target average

### **Composite & Encoding** (4 features)
* `opportunity_score` - snap_share × (touches + targets)
* `fantasy_points_delta` - Week-over-week production change
* `week_number` - Season week (1-18)
* Position encoding: `is_rb`, `is_wr`, `is_te`

### **Training Dataset Stats**
* Total samples: 8,772 player-weeks (2021-2024)
* Breakouts: 67 (0.76% breakout rate)
* Highly imbalanced classification problem

## 🗂️ Available Data Sources for New Features

### **Game Context Tables**
* `team_pace_metrics` - Offensive pace, plays/game, seconds/play
* `game_vegas_totals` - Game totals, spreads, implied team scores
* `defense_weekly_stats` - Opponent defensive strength by position

### **Player Opportunity Tables**
* `player_red_zone_stats` - RZ targets, carries, touchdowns (season-level)
* `player_estimated_routes` - Route participation data
* `player_td_efficiency` - TD conversion rates

### **Situational Tables**
* `silver_injury_reports` - Teammate injury status
* `silver_player_news` - News volume, sentiment, recency
* `silver_rosters` - Depth chart positions

### **Next: Explore each table and engineer features**

## 🧩 How New Features Will Improve Your Ensemble

### **1. Better Model Diversity**

**Current Challenge:**
- Unified and position-specific models use the SAME features
- Low diversity = ensemble gains are limited (only 95-5 optimal weight)
- Both models make similar mistakes

**With New Features:**
- **Unified model** gets contextual features (team pace, opponent, game script)
- **Position-specific models** get position-relevant features:
  - **RB model:** Red zone carries, goal-line role, team run % in RZ
  - **WR model:** Route participation rate, target share trend, air yards
  - **TE model:** Blocking snap %, receiving snap %, inline vs slot usage
- **Result:** Models see different signals → better ensemble performance

### **2. Position-Specific Model Improvements**

| Position | Current Top Features | New Position-Specific Features | Expected Impact |
|----------|---------------------|-------------------------------|----------------|
| **RB** | snap_share_delta, touches | rz_carry_share, goal_line_work, team_run_rate_rz | +15-20% AUC |
| **WR** | targets_delta, snap_share | route_participation_rate, target_share_3wk_trend, air_yards_share | +10-15% AUC |
| **TE** | snap_share, opportunity_score | receiving_snap_%, inline_rate, targets_per_route | +20-25% AUC (most improvement) |

### **3. Situational Breakout Detection**

**New Feature Categories Enable:**
1. **Injury-Driven Breakouts** - Detect when starter goes down
2. **Matchup Breakouts** - Weak opponent defense at position
3. **Game Script Breakouts** - Big underdog = pass volume spike
4. **Role Change Breakouts** - Snap % stable but usage type changes (RZ work increases)

### **4. Expected Ensemble Weight Changes**

**Current:** 95% Unified + 5% Position

**After Enhancement:** Likely 70-75% Unified + 25-30% Position
- Position models gain more unique signal
- Better ensemble diversity
- Higher overall AUC

**Target Performance:**
- Current Ensemble AUC: ~0.79-0.80
- Target Ensemble AUC: 0.85-0.87 (+6-8% improvement)

## ➕ Proposed New Features

### **🎮 Game Context (5 features)**

1. **team_pace_percentile** - Team offensive pace rank (0-100)
   - Fast teams = more plays = more opportunities
   - Use: Join `team_pace_metrics` on team + season

2. **opponent_def_rank_vs_position** - Opponent defense rank vs position (1-32)
   - Weak vs RBs/WRs/TEs = exploitable matchup
   - Use: Join `defense_weekly_stats` on opponent + position

3. **game_total** - Vegas total points for the game
   - High totals = more scoring = more volume
   - Use: Join `game_vegas_totals` on team + week

4. **team_implied_score** - Team's implied points (total + spread)
   - Favorites get more offensive plays
   - Calculate: (game_total + spread) / 2

5. **is_home_game** - Home vs away indicator
   - Home teams typically run more
   - Use: From game schedule data

### **📈 Advanced Usage (5 features)**

6. **target_share_3wk_trend** - Linear regression slope of target % over 3 weeks
   - Trending up = increasing role
   - Calculate: LinearRegression(targets ~ week) for last 3 weeks

7. **snap_share_volatility** - Std dev of snap % over last 3 weeks
   - High volatility = inconsistent role = risky
   - Calculate: STDDEV(snap_share) for last 3 weeks

8. **touches_per_snap** - Efficiency: touches / snaps
   - High = bell cow usage when on field
   - Calculate: (touches + targets) / (snap_share × team_snaps)

9. **rz_opportunity_share** - % of team's RZ touches (last 4 weeks)
   - Goal line work = TD upside
   - Use: `player_red_zone_stats` aggregated

10. **route_participation_rate** - Routes / team pass plays
    - WR/TE involvement in pass game
    - Use: `player_estimated_routes`

### **📄 Situational Context (5 features)**

11. **teammate_injuries_same_position** - Count of injured starters at position
    - Injury vacuum = opportunity spike
    - Use: `silver_injury_reports` filtered by team + position + status IN ('Out', 'Doubtful')

12. **news_mentions_last_7d** - News articles mentioning player (last 7 days)
    - Buzz = coaching staff talking about increased role
    - Use: `silver_player_news` COUNT(*) WHERE date >= current_date - 7

13. **days_since_last_news** - Recency of last news mention
    - Fresh news = current narrative
    - Calculate: current_date - MAX(news_date)

14. **weeks_since_season_start** - Week number standardized
    - Early season (weeks 1-4) behave differently than late (15-18)
    - Calculate: week / 18.0

15. **career_breakout_history** - Has player broken out before? (binary)
    - Lightning strikes twice
    - Use: LEFT JOIN to `historical_breakouts` on player_name

In [0]:
# Load current training data to understand baseline
df_current = spark.table("main.fantasai.breakout_training_data").toPandas()

print("="*80)
print("CURRENT TRAINING DATA SUMMARY")
print("="*80)
print(f"\nTotal samples: {len(df_current):,}")
print(f"Breakouts: {df_current['label'].sum():,} ({df_current['label'].mean():.2%})")
print(f"\nSeasons: {df_current['season'].min()} - {df_current['season'].max()}")
print(f"Weeks: {df_current['week'].min()} - {df_current['week'].max()}")

print(f"\n\nCurrent Feature Set ({len([c for c in df_current.columns if c not in ['season', 'week', 'player_name', 'label']])} features):")
feature_cols = [c for c in df_current.columns if c not in ['season', 'week', 'player_name', 'label']]
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

# Position distribution
print(f"\n\nPosition Distribution:")
for pos, col in [('RB', 'is_rb'), ('WR', 'is_wr'), ('TE', 'is_te')]:
    count = (df_current[col] == 1).sum()
    breakouts = df_current[df_current[col] == 1]['label'].sum()
    rate = df_current[df_current[col] == 1]['label'].mean()
    print(f"  {pos}: {count:,} samples, {int(breakouts)} breakouts ({rate:.2%} rate)")

## ⚠️ CRITICAL ISSUE: Training Data Missing 2025 Season

### **Problem**
Current training data: **2021-2024 only** (8,772 samples, 67 breakouts)

**Missing:** 2025 season data (~2,000+ samples, 18 weeks completed)

### **Why This Matters**
1. **Recency bias** - Model doesn't know about 2025 player usage patterns
2. **Missing breakouts** - 2025 has real breakout examples to learn from
3. **Rule changes** - Any 2025 NFL rule/gameplay changes aren't captured
4. **Roster changes** - New team contexts, coaching changes not reflected

### **Impact on Production Predictions**
- Current model predicts 2025 players using only 2021-2024 patterns
- May miss 2025-specific breakout signals
- Reduced accuracy for current season

### **Action Required**
1. Update `Fantasy Breakout Prediction Engine` notebook to include 2025
2. Regenerate `breakout_training_data` table with 2025 (target: ~10,700 samples)
3. Retrain all models (unified + position-specific + ensemble)
4. Redeploy to production

**Priority:** HIGH - Should be done before adding new features

In [0]:
# Check team pace metrics for contextual features
print("="*80)
print("TEAM PACE METRICS ANALYSIS")
print("="*80)

df_pace = spark.table("main.fantasai.team_pace_metrics").toPandas()

print(f"\nSeasons available: {sorted(df_pace['season'].unique())}")
print(f"Teams: {df_pace['team'].nunique()} unique teams")

print(f"\n\n2024 Top 10 Fastest-Paced Teams:")
df_pace_2024 = df_pace[df_pace['season'] == 2024].sort_values('avg_plays_per_game', ascending=False)
print(df_pace_2024[['team', 'avg_plays_per_game', 'avg_plays_per_minute', 'games_played']].head(10).to_string(index=False))

# Calculate percentiles
df_pace_2024['pace_percentile'] = df_pace_2024['avg_plays_per_game'].rank(pct=True) * 100

print(f"\n\nFeature Engineering Approach:")
print("  1. Calculate team_pace_percentile = PERCENT_RANK() OVER (PARTITION BY season ORDER BY avg_plays_per_game)")
print("  2. High pace (>75th percentile) = more opportunities for breakouts")
print("  3. Join to weekly_usage_features on (team, season)")
print(f"\n  ✅ Data available for seasons: {df_pace['season'].min()}-{df_pace['season'].max()}")
print(f"  ✅ Can add as feature to training data")

In [0]:
# Check red zone stats for position-specific features
print("="*80)
print("RED ZONE USAGE ANALYSIS")
print("="*80)

df_rz_summary = spark.sql("""
    SELECT 
        season,
        position,
        COUNT(DISTINCT player_name) as players,
        ROUND(AVG(rz_total_touches), 1) as avg_rz_touches,
        ROUND(AVG(rz_total_tds), 1) as avg_rz_tds,
        SUM(CASE WHEN rz_total_touches >= 10 THEN 1 ELSE 0 END) as high_rz_usage_count
    FROM main.fantasai.player_red_zone_stats
    WHERE season >= 2021
    GROUP BY season, position
    ORDER BY season DESC, position
""").toPandas()

print(f"\n{df_rz_summary.to_string(index=False)}")

# Sample high RZ usage players
print(f"\n\n2024 High Red Zone Usage Players (10+ touches):")
df_rz_leaders = spark.sql("""
    SELECT player_name, position, team, rz_total_touches, rz_total_tds
    FROM main.fantasai.player_red_zone_stats
    WHERE season = 2024 AND rz_total_touches >= 10
    ORDER BY rz_total_touches DESC
    LIMIT 10
""").toPandas()
print(df_rz_leaders.to_string(index=False))

print(f"\n\n⚠️  Limitation: Red zone data is SEASON-LEVEL, not weekly")
print("  - Can't track week-to-week RZ role changes during season")
print("  - BUT useful for player context: 'Has RZ role' vs 'No RZ role'")
print("\n✅ Feature idea: rz_role_indicator = 1 if player had 5+ RZ touches previous season")

In [0]:
# Check defense stats for matchup features
print("="*80)
print("DEFENSE WEEKLY STATS ANALYSIS")
print("="*80)

# Check schema
df_def_sample = spark.sql("""
    SELECT *
    FROM main.fantasai.defense_weekly_stats
    WHERE season = 2024 AND week = 18
    LIMIT 5
""").toPandas()

print("\nSample Defense Data (Week 18, 2024):")
print(df_def_sample.to_string(index=False))

# Analyze coverage by season
df_def_coverage = spark.sql("""
    SELECT 
        season,
        COUNT(DISTINCT team) as teams,
        COUNT(DISTINCT week) as weeks,
        COUNT(*) as total_records
    FROM main.fantasai.defense_weekly_stats
    GROUP BY season
    ORDER BY season DESC
""").toPandas()

print(f"\n\nDefense Data Coverage:")
print(df_def_coverage.to_string(index=False))

print(f"\n\nFeature Engineering Approach:")
print("  1. Rank defenses by fantasy_points_allowed per position")
print("     - RANK() OVER (PARTITION BY season, week, position ORDER BY fantasy_points_allowed DESC)")
print("  2. Lower rank (1-10) = tough defense, Higher rank (23-32) = weak defense")
print("  3. Join to player weekly data on (opponent_team, week, position)")
print("\n✅ Strong feature for matchup-driven breakouts")

In [0]:
# Check injury and news data availability
print("="*80)
print("INJURY & NEWS DATA ANALYSIS")
print("="*80)

# Injury reports
try:
    df_injury_sample = spark.sql("""
        SELECT player_name, position, team, injury_status, injury_body_part
        FROM main.fantasai.silver_injury_reports
        LIMIT 10
    """).toPandas()
    
    print("\nInjury Report Sample:")
    print(df_injury_sample.to_string(index=False))
    
    injury_count = spark.sql("SELECT COUNT(*) as count FROM main.fantasai.silver_injury_reports").collect()[0]['count']
    print(f"\nTotal injury records: {injury_count:,}")
    print("\n✅ Can use for teammate_injury_impact feature")
    print("   - Count teammates at same position with status IN ('Out', 'Doubtful')")
except Exception as e:
    print(f"\n⚠️  Injury data error: {e}")

print("\n" + "-"*80)

# Player news
try:
    df_news_sample = spark.sql("""
        SELECT player_name, title, published_date
        FROM main.fantasai.silver_player_news
        ORDER BY published_date DESC
        LIMIT 5
    """).toPandas()
    
    print("\nRecent Player News Sample:")
    print(df_news_sample.to_string(index=False))
    
    news_count = spark.sql("SELECT COUNT(*) as count FROM main.fantasai.silver_player_news").collect()[0]['count']
    print(f"\nTotal news records: {news_count:,}")
    print("\n✅ Can use for news-based features:")
    print("   - news_mentions_last_7d: COUNT(*) WHERE published_date >= current_date - 7")
    print("   - days_since_last_news: DATEDIFF(current_date, MAX(published_date))")
except Exception as e:
    print(f"\n⚠️  News data error: {e}")

In [0]:
# Check weather data for API chat integration
print("="*80)
print("WEATHER DATA ANALYSIS (for API Chat)")
print("="*80)

try:
    df_weather_sample = spark.sql("""
        SELECT *
        FROM main.fantasai.bronze_nfl_weather
        ORDER BY game_date DESC
        LIMIT 5
    """).toPandas()
    
    print("\nRecent Weather Data Sample:")
    print(df_weather_sample.to_string(index=False))
    
    weather_count = spark.sql("SELECT COUNT(*) as count FROM main.fantasai.bronze_nfl_weather").collect()[0]['count']
    weather_seasons = spark.sql("SELECT MIN(season) as min_season, MAX(season) as max_season FROM main.fantasai.bronze_nfl_weather").collect()[0]
    
    print(f"\nTotal weather records: {weather_count:,}")
    print(f"Seasons covered: {weather_seasons['min_season']} - {weather_seasons['max_season']}")
    print("\n✅ Weather data available")
    print("   - Should be added to API chat for game context")
    print("   - Can inform recommendations (e.g., bad weather = run-heavy)")
except Exception as e:
    print(f"\n⚠️  Weather data error: {e}")
    print("   - May need to check table name or schema")

## 🏈 YAC (Yards After Catch) Analysis

**Why YAC Matters for Breakouts:**
- High YAC = playmaking ability, big play potential
- YAC efficiency predicts volume increases
- WR/TE breakouts often preceded by YAC spikes
- RB pass-catchers with high YAC get more targets

**Features to Create:**
1. `yac_per_reception` - Efficiency metric
2. `yac_percentage` - YAC / total receiving yards
3. `yac_delta` - Week-over-week change
4. `avg_yac_prev_3wk` - Rolling average

In [0]:
# Check if YAC (Yards After Catch) data exists
import re

print("="*80)
print("CHECKING FOR YAC DATA")
print("="*80)

# Check gold_weekly_stats columns
print("\n1. Checking gold_weekly_stats columns:")
cols_gold = spark.table("main.fantasai.gold_weekly_stats").columns
yac_cols = [c for c in cols_gold if 'yac' in c.lower() or 'after' in c.lower() or 'air' in c.lower()]
print(f"   YAC/Air-related columns: {yac_cols if yac_cols else 'None found'}")

# Check weekly_usage_features
print("\n2. Checking weekly_usage_features columns:")
cols_usage = spark.table("main.fantasai.weekly_usage_features").columns
yac_cols_usage = [c for c in cols_usage if 'yac' in c.lower() or 'after' in c.lower() or 'air' in c.lower()]
print(f"   YAC/Air-related columns: {yac_cols_usage if yac_cols_usage else 'None found'}")

# Sample receiving data to see what we have
print("\n3. Sample 2024 WR receiving data (Week 18):")
df_receiving = spark.sql("""
    SELECT player_name, position, team,
           receptions, receiving_yards, targets,
           ROUND(receiving_yards / NULLIF(receptions, 0), 1) as yards_per_reception
    FROM main.fantasai.gold_weekly_stats
    WHERE season = 2024 AND week = 18 AND position = 'WR'
          AND receptions > 0
    ORDER BY receiving_yards DESC
    LIMIT 5
""").toPandas()
print(df_receiving.to_string(index=False))

print("\n" + "="*80)
if yac_cols or yac_cols_usage:
    print("✅ YAC DATA FOUND - Can add directly to features")
    print("\nNext: Calculate YAC features and add to training data")
else:
    print("⚠️  NO YAC DATA FOUND in current tables")
    print("\nOptions:")
    print("  1. Source YAC from nfl_data_py (nflverse)")
    print("  2. Use yards_per_reception as proxy")
    print("  3. Calculate if air_yards available: YAC = receiving_yards - air_yards")
    print("\nRecommendation: Import from nflverse alongside player avatars")

## 🖼️ Player Avatar/Headshot Integration

**Purpose:** Enhance API responses with player headshot images from nflverse

**Data Source:** `nfl_data_py` Python package (nflverse)

**Key Fields:**
- `gsis_id` - NFL GSIS unique player ID
- `player_name` - Full player name
- `position` - Position
- `team` - Current team abbreviation
- `headshot` - URL to player headshot image

**Use Cases:**
1. API chat responses include player images
2. R2 export includes headshot URLs
3. UI can display player photos with predictions

**Next: Import player data and store in Unity Catalog**

In [0]:
# Import nflverse player data including headshots
import nfl_data_py as nfl
import pandas as pd

print("="*80)
print("IMPORTING NFLVERSE PLAYER DATA")
print("="*80)

print("\nImporting player roster data with headshots...")
try:
    players = nfl.import_players()
    
    # Filter to active offensive players
    offensive_positions = ['QB', 'RB', 'WR', 'TE', 'FB']
    players_active = players[
        (players['position'].isin(offensive_positions)) &
        (players['status'] == 'ACT')  # Active players only
    ].copy()
    
    print(f"\n✅ Successfully imported {len(players_active):,} active offensive players")
    
    # Show sample
    print("\nSample player data:")
    cols_display = ['gsis_id', 'player_name', 'position', 'team', 'headshot']
    print(players_active[cols_display].head(10).to_string(index=False))
    
    # Save to Unity Catalog
    print("\n\nSaving to Unity Catalog: main.fantasai.nflverse_players")
    df_players = spark.createDataFrame(players_active)
    df_players.write.mode("overwrite").saveAsTable("main.fantasai.nflverse_players")
    
    print("✅ Player data with headshots saved successfully")
    print(f"\nTable: main.fantasai.nflverse_players")
    print(f"Columns: {', '.join(df_players.columns)}")
    
    # Check headshot coverage
    headshot_count = players_active['headshot'].notna().sum()
    headshot_pct = headshot_count / len(players_active) * 100
    print(f"\nHeadshot coverage: {headshot_count}/{len(players_active)} ({headshot_pct:.1f}%)")
    
except Exception as e:
    print(f"\n❌ Error importing player data: {e}")
    print("\nTroubleshooting:")
    print("  1. Ensure nfl_data_py is installed: %pip install nfl_data_py")
    print("  2. Check internet connectivity for data download")
    print("  3. Retry import after a few minutes")

## 📊 Last 3 Games Sparkline Feature

**Purpose:** Visual representation of recent player performance trend

**Logic:**
1. Fetch last 3 games of fantasy points for current season
2. If player has <3 games in current season:
   - Backfill from previous season's last weeks
   - Example: 2025 Week 2 player → use [2024 W18, 2024 W17, 2025 W1]
3. Store as JSON array: `[12.5, 18.3, 22.1]`
4. Return with API responses for UI rendering

**Use Cases:**
1. API chat includes sparkline in player summaries
2. Breakout alerts show trend visualization
3. R2 export includes sparkline data for frontend

**Next: Implement sparkline generation function**

In [0]:
# Function to generate last 3 games sparkline data
from pyspark.sql import functions as F, Window

def get_last_3_games_sparkline(player_name, current_season, current_week):
    """
    Get last 3 games fantasy points for sparkline visualization.
    Backfills from previous season if needed.
    
    Returns: List of 3 fantasy point values, chronologically ordered
    """
    
    # Get player's recent games across seasons
    df_recent = spark.sql(f"""
        WITH player_games AS (
            SELECT 
                season,
                week,
                player_name,
                fantasy_points,
                ROW_NUMBER() OVER (
                    PARTITION BY player_name 
                    ORDER BY season DESC, week DESC
                ) as game_recency_rank
            FROM main.fantasai.gold_weekly_stats
            WHERE player_name = '{player_name}'
                  AND season IN ({current_season}, {current_season - 1})
                  AND fantasy_points IS NOT NULL
        )
        SELECT season, week, fantasy_points, game_recency_rank
        FROM player_games
        WHERE game_recency_rank <= 3
        ORDER BY season ASC, week ASC
    """)
    
    games = df_recent.collect()
    
    if len(games) >= 3:
        # Have at least 3 games - take the 3 most recent
        sparkline = [float(g['fantasy_points']) for g in games[-3:]]
    elif len(games) > 0:
        # Have some games but <3 - pad with nulls
        sparkline = [float(g['fantasy_points']) for g in games]
        sparkline = [None] * (3 - len(sparkline)) + sparkline  # Pad front with nulls
    else:
        # No games found
        sparkline = [None, None, None]
    
    return sparkline

# Test the function
print("="*80)
print("SPARKLINE GENERATION TEST")
print("="*80)

# Test with a known active player
test_players = ['Ja\'Marr Chase', 'Christian McCaffrey', 'Travis Kelce']
current_season = 2024
current_week = 18

for player in test_players:
    sparkline = get_last_3_games_sparkline(player, current_season, current_week)
    print(f"\n{player}:")
    print(f"  Last 3 games: {sparkline}")
    if all(x is not None for x in sparkline):
        trend = "↗️" if sparkline[2] > sparkline[0] else "↘️" if sparkline[2] < sparkline[0] else "→"
        print(f"  Trend: {trend}")

print("\n" + "="*80)
print("✅ Sparkline function ready")
print("\nNext steps:")
print("  1. Add sparkline generation to API chat retrieval")
print("  2. Return sparkline array with player predictions")
print("  3. Frontend can render using mini line charts or spark bars")

## 📋 UPDATED Proposed New Features

### **Original 15 Features:**
1. team_pace_percentile
2. opponent_def_rank_vs_position  
3. game_total
4. team_implied_score
5. is_home_game
6. target_share_3wk_trend
7. snap_share_volatility
8. touches_per_snap
9. rz_opportunity_share
10. route_participation_rate
11. teammate_injuries_same_position
12. news_mentions_last_7d
13. days_since_last_news
14. weeks_since_season_start
15. career_breakout_history

---

### **🆕 NEW Position-Specific Features (YAC):**

**16. yac_per_reception** (WR/TE/RB)
- YAC / receptions
- Measures playmaking ability after catch
- High YAC = explosive player = breakout candidate

**17. yac_percentage** (WR/TE/RB)  
- YAC / receiving_yards
- Shows % of yards created after catch vs air yards
- High % = YAC specialist, low % = downfield threat

**18. yac_delta** (WR/TE/RB)
- Week-over-week YAC change
- Sudden YAC spike = scheme change or increased opportunity

**19. avg_yac_prev_3wk** (WR/TE/RB)
- Rolling 3-week YAC average
- Context for current week YAC

---

### **🆕 API/UI Enhancement Features (Not ML Features):**

**Player Avatar/Headshot:**
- Store: `main.fantasai.nflverse_players`
- Join on: player_name + team
- Return: headshot URL with all API responses

**Last 3 Games Sparkline:**
- Calculate: Last 3 fantasy point values
- Backfill: From previous season if <3 current season games
- Return: JSON array `[fp1, fp2, fp3]` for UI visualization

---

### **Total Enhanced Feature Count:**
- **ML Training Features:** 13 current + 19 new = **32 features**
- **API/UI Enhancements:** Player avatars + sparklines
- **Expected Impact:** Ensemble AUC 0.79 → 0.86-0.88 (+8-10%)

## 📊 Section 4: Feature Importance Analysis

### **Approach**
1. Train baseline model with current 13 features
2. Calculate feature importance from Gradient Boosting
3. Identify top predictors
4. Compare against proposed new features

### **Current Top Predictors (from existing model)**
1. **snap_share_delta** (27%) - Biggest usage change predictor
2. **opportunity_score** (18%) - Composite usage metric
3. **avg_snap_share_prev_2wk** (12%) - Historical context
4. **targets_delta** (8%) - Target share changes
5. **snap_share** (7%) - Current usage level

### **Next: Test new features and compare**

In [0]:
# Demonstrate how to calculate a few new features
print("="*80)
print("SAMPLE FEATURE ENGINEERING")
print("="*80)

# Example: Calculate snap_share_volatility (3-week rolling std dev)
df_feature_demo = spark.sql("""
    WITH player_weekly AS (
        SELECT 
            season,
            week,
            player_name,
            position,
            team,
            snap_share,
            targets,
            -- 3-week rolling std dev of snap share
            STDDEV(snap_share) OVER (
                PARTITION BY player_name, season 
                ORDER BY week 
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) as snap_share_volatility,
            -- 3-week rolling avg of targets (trend)
            AVG(targets) OVER (
                PARTITION BY player_name, season 
                ORDER BY week 
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) as target_share_3wk_avg
        FROM main.fantasai.weekly_usage_features
        WHERE season = 2024 AND week >= 3  -- Need 3 weeks for rolling calcs
    )
    SELECT 
        player_name,
        position,
        team,
        week,
        ROUND(snap_share, 3) as snap_share,
        ROUND(snap_share_volatility, 3) as volatility,
        ROUND(target_share_3wk_avg, 1) as targets_3wk_avg
    FROM player_weekly
    WHERE week = 18  -- Most recent week
        AND snap_share > 0.5  -- High snap share players
    ORDER BY snap_share_volatility DESC NULLS LAST
    LIMIT 10
""").toPandas()

print("\nTop 10 Players by Snap Share Volatility (Week 18, 2024):")
print("High volatility = inconsistent role = risky for breakout predictions")
print()
print(df_feature_demo.to_string(index=False))

print("\n\n✅ Example shows how to calculate rolling statistics for trend features")
print("✅ Can apply same approach for: target_share_trend, touches_volatility, etc.")

## 🗺️ Section 5: Implementation Roadmap

### **Phase 1: Update Training Data with 2025 (CRITICAL - Do First)**
**Timeline:** 1-2 days

1. Open [Fantasy Breakout Prediction Engine](#notebook-1202378217801264)
2. Update date filters to include 2025 season
3. Re-run all cells to regenerate breakout labels
4. Verify 2025 breakouts identified correctly
5. Save updated `breakout_training_data` table (~10,700 samples expected)
6. Retrain models with 2025 data

**Expected Impact:** +5-10% AUC improvement just from recency

---

### **Phase 2: Add Core Contextual Features (High Value, Low Effort)**
**Timeline:** 2-3 days

**Features to add first (best ROI):**
1. ✅ `team_pace_percentile` - Data ready, simple join
2. ✅ `opponent_def_rank_vs_position` - Data ready, needs ranking calc
3. ✅ `teammate_injuries_same_position` - Data ready, needs aggregation
4. ✅ `snap_share_volatility` - Easy rolling calc
5. ✅ `breakout_history` - Simple lookup from historical_breakouts

**Steps:**
1. Create new notebook: `Feature Engineering - Enhanced`
2. Write SQL to calculate each feature
3. Join to `weekly_usage_features`
4. Save as `breakout_training_data_enhanced`
5. Train models on both old and new feature sets
6. Compare AUC/precision/recall

---

### **Phase 3: Add Position-Specific Features (Ensemble Boost)**
**Timeline:** 3-4 days

**RB-specific:**
- Red zone carry share
- Goal line work indicator
- Team run rate in RZ

**WR-specific:**
- Route participation rate
- Target share 3-week trend
- Deep target %

**TE-specific:**
- Receiving snap %
- Inline vs slot usage
- Targets per route run

**Expected Impact:** Position-specific models improve 15-25% AUC

---

### **Phase 4: Production Deployment & A/B Test**
**Timeline:** 1 week

1. Register enhanced models in Unity Catalog
2. Update production prediction notebook
3. Run parallel predictions (old vs new models) for 2 weeks
4. Compare breakout detection rates
5. Switch to new model if performance better
6. Update R2 export with enhanced predictions

---

### **Phase 5: Update API Chat Data Access**
**Timeline:** 1-2 days

**Add to API chat retrieval function:**
1. Breakout predictions from `breakout_predictions_current`
2. Opportunity scores from `weekly_usage_features`
3. Weather data from `bronze_nfl_weather`
4. Defense rankings from `defense_rankings_current`

**Location:** `/Repos/.../06_Exports/fantasai_chat_api_deployment`

---

### **Total Timeline: 2-3 weeks**

## 🎯 Section 6: Expected Improvements & Success Metrics

### **Current Baseline Performance**
- **Ensemble AUC:** ~0.79-0.80
- **Precision:** ~5-8% (many false positives)
- **Recall:** ~50-60% (missing half of real breakouts)
- **Breakout rate:** 0.76% (67 / 8,772 samples)

---

### **Phase 1 Expected Impact: Add 2025 Data**
**Estimated Improvement:**
- **Ensemble AUC:** 0.82-0.84 (+3-5%)
- **Recall:** 65-70% (catch 2 out of 3 breakouts)
- **Why:** Model learns from most recent season, better captures current NFL trends

---

### **Phase 2 Expected Impact: Core Contextual Features**
**Estimated Improvement:**
- **Ensemble AUC:** 0.84-0.86 (+2-3% additional)
- **Precision:** 10-12% (fewer false alarms)
- **Recall:** 70-75% (catch 3 out of 4 breakouts)
- **Why:** Context-aware (pace, opponent, injuries) reduces noise

**Key Features Expected Impact:**
1. `opponent_def_rank_vs_position` - Identifies matchup breakouts (+2% recall)
2. `teammate_injuries_same_position` - Catches injury-driven breakouts (+3% recall)
3. `snap_share_volatility` - Filters out unstable roles (reduces false positives)

---

### **Phase 3 Expected Impact: Position-Specific Features**
**Estimated Improvement:**
- **Ensemble AUC:** 0.85-0.87 (+1-2% additional)
- **Position-specific model weight:** Increases from 5% to 25-30%
- **Recall:** 75-80% (catch 4 out of 5 breakouts)
- **Why:** Position models gain unique signal, better ensemble diversity

**Position-Specific Gains:**
- **RB model:** +15-20% AUC (RZ features are strong)
- **WR model:** +10-15% AUC (route/target trends help)
- **TE model:** +20-25% AUC (biggest gains - currently weak)

---

### **Final Target Performance (All Phases Complete)**

| Metric | Current | Target | Improvement |
|--------|---------|--------|-------------|
| **Ensemble AUC** | 0.79 | 0.85-0.87 | +6-8% |
| **Precision @ 10%** | 5-8% | 10-15% | +2x |
| **Recall** | 50-60% | 75-80% | +25-30% |
| **Early Detection** | Same week | 1-2 weeks early | Timing shift |

---

### **Success Criteria**

✅ **Phase 1 Success:**
- Training data includes 2025 season
- Model AUC improves by at least 3%
- No regression on 2024 test set

✅ **Phase 2 Success:**
- Precision reaches 10%+ (1 real breakout per 10 alerts)
- Recall reaches 70%+ (catch 7 out of 10 breakouts)
- Production predictions use enhanced features

✅ **Phase 3 Success:**
- Ensemble weight shifts to 70-30 or 75-25 (unified-position)
- Position-specific models show clear specialized performance
- Overall ensemble AUC >0.85

✅ **Production Success:**
- A/B test shows new model outperforms old by 10%+ in real breakout detection
- User feedback: waiver wire alerts are more actionable
- API chat integrates breakout scores, opportunity metrics, weather context

---

### **Monitoring & Iteration**
- Track weekly: breakout alerts vs actual breakouts
- Monthly review: false positive rate, missed breakouts
- Quarterly: retrain with new season data
- Annual: major feature engineering refresh

## ✅ Next Steps & Immediate Action Items

### **CRITICAL: Address Missing 2025 Data First**

**Before adding new features, you MUST update training data with 2025:**

1. **Open:** [Fantasy Breakout Prediction Engine](#notebook-1202378217801264)
2. **Update:** Date filters to include season = 2025
3. **Run:** All cells to regenerate breakout labels with 2025
4. **Verify:** Breakout count increases from 67 to ~90-100 (adding 2025 breakouts)
5. **Check:** Training samples increase from 8,772 to ~10,700+
6. **Retrain:** Models with expanded dataset

**Why this matters:** Your production model is predicting 2025 players using only 2021-2024 patterns. This is like predicting the future without knowing the present.

---

### **High Priority: Update API Chat Data Access**

Your API chat is missing critical data sources:

**Edit:** `/Repos/.../06_Exports/fantasai_chat_api_deployment`

**Add retrieval functions for:**
1. ✅ Breakout predictions: Query `main.fantasai.breakout_predictions_current`
2. ✅ Opportunity scores: Query `main.fantasai.weekly_usage_features`
3. ✅ Weather data: Query `main.fantasai.bronze_nfl_weather`
4. ✅ Defense rankings: Query `main.fantasai.defense_rankings_current`

**Currently:** API only queries `gold_weekly_stats` (basic fantasy points)

**After fix:** API can answer:
- "Who are the top breakout candidates this week?"
- "Show me players with high opportunity scores"
- "What's the weather for Sunday's games?"
- "Which defenses are weak against WRs?"

---

### **Quick Wins (Can Do Today)**

1. **Run this notebook's Cell 6** to see current training data stats
2. **Run Cells 7-11** to explore available data sources
3. **Review** the implementation roadmap (Cell 13)
4. **Decide:** Tackle 2025 training data first, or API chat updates first?

---

### **Recommended Order**

**Option A: Data-First Approach (Recommended)**
1. Fix 2025 training data (1-2 days)
2. Retrain models (1 day)
3. Add core features (2-3 days)
4. Update API chat (1 day)
5. Deploy enhanced models (1 week)

**Option B: User-Facing First**
1. Update API chat (1 day) - immediate user impact
2. Fix 2025 training data (1-2 days)
3. Add features + retrain (3-4 days)
4. Deploy (1 week)

---

**Decision Point:** Which should we prioritize first?

[Fix 2025 training data](#followup) [Update API chat access](#followup) [Do both in parallel](#followup)